# DEE — Termodinámica del estado fundamental del cristal

**Objetivo:** caracterizar cuantitativamente el estado de vacío del cristal DEE como objeto físico, calculando sus cuatro propiedades termodinámicas fundamentales:

1. **Tensión cuántica residual** — energía elástica almacenada en las fluctuaciones de punto cero
2. **Densidad de energía y masa equivalente** — refinamiento del cálculo anterior
3. **Presión del vacío** — verificación de la ecuación de estado P = w·ρc²
4. **Módulo de bulk K** — resistencia del cristal a compresión uniforme

Estas son las cantidades físicas que cualquier cristal real tiene en su estado fundamental cuántico. Si DEE es genuinamente un cristal elástico, estas cantidades deben tener valores bien definidos.

**Pregunta abierta:** ¿hay relaciones internas entre estas cantidades que sugieran principios de equilibrio?

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_termo_vacio'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

# Constantes físicas
c_SI = 2.998e8
hbar_SI = 1.055e-34
G_SI = 6.674e-11
k_B_SI = 1.381e-23
ell_P = np.sqrt(hbar_SI * G_SI / c_SI**3)
rho_P = c_SI**5 / (hbar_SI * G_SI**2)

print('Setup listo')

In [ ]:
# Funciones del cristal (idénticas a notebooks anteriores)
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def build_laplacian(D_mat, k_neighbors=30, alpha=2.0):
    N = len(D_mat)
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                rows.append(i); cols.append(j); vals.append(1.0/d**alpha)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    return diags(degrees) - W

# Construir cristal de referencia
N_TARGET = 1331
JITTER = 0.1
K_NEIGHBORS = 30

points, N = grid_perturbed_T3(N_TARGET, JITTER, seed=42)
D_mat = periodic_distance_matrix(points, L=1.0)
L_op = build_laplacian(D_mat, K_NEIGHBORS, alpha=2.0)

# Diagonalizar para obtener espectro completo (o casi completo)
N_EIGS = min(800, 3*N - 5)
print(f'Diagonalizando {N_EIGS} autovalores (de {3*N} totales)...')
t0 = time.time()
eigs_all, eigvecs_all = eigsh(L_op, k=N_EIGS, which='SM', tol=1e-8)
idx_sort = np.argsort(eigs_all)
eigs_all = eigs_all[idx_sort]
eigvecs_all = eigvecs_all[:, idx_sort]
print(f'Tiempo: {time.time()-t0:.1f}s')

eigs_nonzero = eigs_all[eigs_all > 1e-3]
vecs_nonzero = eigvecs_all[:, eigs_all > 1e-3]
omegas_nat = np.sqrt(eigs_nonzero)
n_modes = len(omegas_nat)

# Velocidad del sonido (del primer modo)
k1_natural = 2 * np.pi  # primer modo |n|=1 en cubo unidad
c_s_natural = omegas_nat[0] / k1_natural

print(f'\nN = {N} nodos')
print(f'n_modes calculados = {n_modes}')
print(f'Velocidad del sonido (natural): c_s = {c_s_natural:.4f}')
print(f'Frecuencias: ω = [{omegas_nat.min():.3f}, {omegas_nat.max():.3f}]')

## Cantidad 1 — Tensión cuántica residual (fluctuación de punto cero por nodo)

En un cristal armónico cuántico, cada modo n tiene una fluctuación de amplitud cuadrática media en su estado fundamental:

$$\langle \hat\phi_n^2 \rangle = \frac{\hbar}{2 m \omega_n}$$

donde m es la "masa efectiva" del modo. En unidades naturales (m=ℏ=1), esto se reduce a 1/(2ω_n).

La **fluctuación cuadrática total por nodo i** (sumada sobre todos los modos) es:

$$\sigma_i^2 = \sum_n |v_n(i)|^2 \frac{1}{2\omega_n}$$

Esta es la "tensión cuántica" del nodo en el vacío — cuánto se desplaza por incertidumbre, aunque no haya excitaciones reales.

La **energía elástica residual** asociada se calcula desde σ_i² y las constantes de fuerza.

In [ ]:
# Fluctuación cuadrática por nodo
# σ_i² = Σ_n |v_n(i)|² / (2ω_n)

weights_nodes = 1.0 / (2 * omegas_nat)  # peso de cada modo: 1/(2ω)

# vecs_nonzero tiene shape (N, n_modes)
# |v_n(i)|² es vecs_nonzero[i, n]²
sigma_sq_per_node = np.sum(vecs_nonzero**2 * weights_nodes[None, :], axis=1)

sigma_sq_mean = np.mean(sigma_sq_per_node)
sigma_sq_std = np.std(sigma_sq_per_node)

# Fluctuación lineal (no cuadrática) — la "amplitud típica" de fluctuación
sigma_typical = np.sqrt(sigma_sq_mean)

print(f'Tensión cuántica del estado fundamental (unidades naturales):')
print(f'  ⟨σ²⟩ por nodo (medio):    {sigma_sq_mean:.4f}')
print(f'  ⟨σ²⟩ desviación entre nodos: {sigma_sq_std:.4f}')
print(f'  Amplitud típica σ:        {sigma_typical:.4f}')
print(f'  Coef. variación:           {sigma_sq_std/sigma_sq_mean*100:.1f}%')
print()

# Comparar con el espaciado típico entre nodos
spacing_typical = 1.0 / (N**(1/3))  # espaciado en unidades de cubo unidad
print(f'  Espaciado típico entre nodos: a = {spacing_typical:.4f}')
print(f'  Razón σ/a:                    {sigma_typical/spacing_typical:.4f}')
print(f'  (σ/a > 1 ⇒ desplazamiento cuántico > distancia entre nodos: PROBLEMA)')
print(f'  (σ/a < 1 ⇒ desplazamiento cuántico subdominante: OK)')

In [ ]:
# Visualización: distribución espacial de la tensión cuántica
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de σ² por nodo
ax = axes[0]
ax.hist(sigma_sq_per_node, bins=40, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(sigma_sq_mean, color='red', linestyle='--', lw=2, label=f'⟨σ²⟩ = {sigma_sq_mean:.3f}')
ax.set_xlabel('σ² (fluctuación cuántica² por nodo)', fontsize=12)
ax.set_ylabel('Número de nodos', fontsize=12)
ax.set_title('Distribución de tensión cuántica entre nodos', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Mapa 3D: visualizar cuáles nodos tienen más tensión
from mpl_toolkits.mplot3d import Axes3D
ax = fig.add_subplot(1, 2, 2, projection='3d')
sc = ax.scatter(points[:, 0], points[:, 1], points[:, 2],
                 c=sigma_sq_per_node, s=30, cmap='viridis', alpha=0.7)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Distribución espacial de la tensión cuántica', fontsize=11)
plt.colorbar(sc, ax=ax, shrink=0.6, label='σ²')

plt.tight_layout()
save_plot('37_tension_cuantica')
plt.show()

## Cantidad 2 — Densidad de energía del estado fundamental (refinada)

Recalculamos E_0 por modo y por unidad de volumen, esta vez con la fracción correcta de modos del espectro completo.

In [ ]:
# Energía de punto cero
E_0_per_mode = 0.5 * omegas_nat
E_0_total_natural = np.sum(E_0_per_mode)
E_0_per_node_natural = E_0_total_natural / N

# Modos calculados / modos totales
frac_modes_calc = n_modes / (3 * N)

# Densidad de energía en unidades naturales (V_natural = 1)
rho_E_natural = E_0_total_natural / 1.0

print(f'Energía de punto cero (unidades naturales):')
print(f'  E_0 total (modos calculados):   {E_0_total_natural:.2f}')
print(f'  E_0 por nodo:                    {E_0_per_node_natural:.4f}')
print(f'  E_0 por modo (promedio):        {np.mean(E_0_per_mode):.4f}')
print(f'  Fracción modos calculados:       {frac_modes_calc*100:.1f}% de 3N totales')
print()
print(f'  Densidad de energía (V_nat=1):  ρ_E = {rho_E_natural:.2f}')

## Cantidad 3 — Presión del vacío y ecuación de estado

En un cristal armónico isótropo en 3D, la energía total y la presión están relacionadas vía la **ecuación de estado fonónica**. Para fonones acústicos:

$$P = -\frac{1}{3} \rho c_s^2 \cdot (1 + \text{correcciones})$$

El parámetro de ecuación de estado es:

$$w = \frac{P}{\rho c_s^2}$$

Para un fluido de fonones acústicos (radiación de sonido), w = +1/3.  
Para un "sólido" en su estado fundamental, w → -1 (cosmológicamente, energía oscura).

Calculamos w directamente a partir del espectro usando relaciones termodinámicas.

**Cálculo de presión vía teorema del virial cuántico:**

$$P = -\frac{\partial E_0}{\partial V}\bigg|_{\text{adiab}}$$

Como ω_n ∝ V^(-γ) con γ el parámetro de Grüneisen (~1 para cristales acústicos), y E_0 ∝ Σω_n:

In [ ]:
# Cálculo de presión vía deformación adiabática
# Construimos cristales con volúmenes ligeramente distintos y medimos cómo cambia E_0

# Escalas de volumen: factor (1 + ε) en las distancias
# Si todas las d_ij → (1+ε) d_ij, entonces w_ij = 1/d² → 1/(1+ε)² · w_ij
# Entonces L → 1/(1+ε)² L  y  ω_n → 1/(1+ε) · ω_n
# E_0 = Σ½ω_n → 1/(1+ε) · E_0
# V = (1+ε)³ · V₀
# P = -dE/dV = -(dE/dε) / (dV/dε) = -[-E₀/(1+ε)²] / [3(1+ε)²·V₀]
#    = E₀ / [3 · (1+ε)⁴ · V₀]
# En ε=0:  P = E₀/(3·V₀)

# Sin embargo este cálculo asume kernel rígido 1/d². 
# Verifiquémoslo numéricamente cambiando V y midiendo E_0

epsilons = [-0.02, -0.01, 0.0, 0.01, 0.02]
E_0_vs_eps = []
V_vs_eps = []

print('Calculando E_0(V) para extraer presión...')

for eps in epsilons:
    # Reescalear puntos: si distancias → (1+ε), entonces el cubo es (1+ε) por lado
    L_box = 1.0 + eps  
    points_eps = points * L_box  # los puntos se reescalan junto con el cubo
    
    # Recalcular distancias con la nueva caja
    D_eps = periodic_distance_matrix(points_eps, L=L_box)
    L_op_eps = build_laplacian(D_eps, K_NEIGHBORS, alpha=2.0)
    
    # Diagonalizar (menos modos para ahorrar tiempo)
    eigs_eps, _ = eigsh(L_op_eps, k=400, which='SM', tol=1e-8)
    eigs_eps = np.sort(eigs_eps)
    omegas_eps = np.sqrt(eigs_eps[eigs_eps > 1e-3])
    
    E_0_eps = np.sum(0.5 * omegas_eps)
    V_eps = L_box**3
    
    E_0_vs_eps.append(E_0_eps)
    V_vs_eps.append(V_eps)
    
    print(f'  ε = {eps:+.3f}: V = {V_eps:.4f}, E₀ = {E_0_eps:.2f}')

E_0_vs_eps = np.array(E_0_vs_eps)
V_vs_eps = np.array(V_vs_eps)

# Calcular dE/dV por diferencias finitas alrededor de ε=0
idx_0 = epsilons.index(0.0)
dE_dV = (E_0_vs_eps[idx_0+1] - E_0_vs_eps[idx_0-1]) / (V_vs_eps[idx_0+1] - V_vs_eps[idx_0-1])
P_vacuum = -dE_dV

rho_E = E_0_vs_eps[idx_0] / V_vs_eps[idx_0]

# Parámetro de ecuación de estado w = P / (ρ c_s²)
w_eos = P_vacuum / (rho_E * c_s_natural**2)

print(f'\nResultados termodinámicos:')
print(f'  ρ_E (densidad de energía):     {rho_E:.4f}')
print(f'  P_vac (presión del vacío):     {P_vacuum:.4f}')
print(f'  c_s² (vel. sonido al cuadrado): {c_s_natural**2:.4f}')
print(f'  w = P/(ρc²):                    {w_eos:.4f}')
print()
print(f'Interpretación de w:')
print(f'  w = +1/3:  fluido de fonones (radiación de sonido)')
print(f'  w = 0:     polvo no relativista')
print(f'  w = -1/3:  límite "cualquier expansión" (cosmología)')
print(f'  w = -1:    energía oscura pura')

## Cantidad 4 — Módulo de bulk K (resistencia a compresión)

El módulo de bulk K mide cuánta presión se requiere para producir una compresión fraccionaria dada:

$$K = -V \frac{\partial P}{\partial V} = V \frac{\partial^2 E_0}{\partial V^2}$$

Para un cristal isótropo, K se relaciona con c_s y la densidad por:

$$c_s^2 = \frac{K}{\rho}$$

Esto nos da una **verificación consistencia**: si calculamos K independientemente y comparamos con ρ·c_s², deben coincidir.

In [ ]:
# Segunda derivada de E₀ respecto a V
# Aproximación de diferencias finitas centrada
if len(epsilons) >= 5:
    dV = V_vs_eps[idx_0+1] - V_vs_eps[idx_0]  # paso en V
    d2E_dV2 = (E_0_vs_eps[idx_0+1] + E_0_vs_eps[idx_0-1] - 2*E_0_vs_eps[idx_0]) / dV**2
    K_bulk = V_vs_eps[idx_0] * d2E_dV2
    
    # Verificación: K debería igualar ρ·c²
    K_predicted = rho_E * c_s_natural**2
    
    print(f'Módulo de bulk K (resistencia a compresión):')
    print(f'  K (medido)     = {K_bulk:.4f}')
    print(f'  ρ · c_s²       = {K_predicted:.4f}')
    print(f'  Razón medido/predicho: {K_bulk/K_predicted:.3f}')
    print()
    if abs(K_bulk/K_predicted - 1) < 0.3:
        print(f'  → ✓ Relación termodinámica K = ρc² confirmada')
    else:
        print(f'  → Diferencia significativa, indica corrección anharmónica o de borde')
else:
    K_bulk = None
    print('No hay suficientes puntos para calcular K')

In [ ]:
# Plot: E₀(V) y dependencia funcional
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# E₀ vs V
ax = axes[0]
ax.plot(V_vs_eps, E_0_vs_eps, 'o', markersize=12, color='steelblue')
# Ajuste polinomial
from numpy.polynomial import polynomial as P
coefs = np.polyfit(V_vs_eps, E_0_vs_eps, 2)
V_smooth = np.linspace(V_vs_eps.min(), V_vs_eps.max(), 100)
ax.plot(V_smooth, np.polyval(coefs, V_smooth), '-', color='red', lw=2,
        label=f'Ajuste cuadrático')
ax.set_xlabel('V (volumen)', fontsize=12)
ax.set_ylabel('E₀ (energía de punto cero)', fontsize=12)
ax.set_title('E₀(V) — pendiente = -P, curvatura = K/V', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# w vs algún parámetro (acá solo como punto de referencia)
ax = axes[1]
categories = ['Radiación\n(w=1/3)', 'Polvo\n(w=0)', 'Frontera\n(w=-1/3)', 
              'Energía osc.\n(w=-1)', f'DEE vacío\n(w={w_eos:+.2f})']
values = [1/3, 0, -1/3, -1, w_eos]
colors = ['orange', 'gray', 'purple', 'green', 'crimson']
ax.barh(categories, values, color=colors, alpha=0.7)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Parámetro de ecuación de estado w = P/(ρc²)', fontsize=12)
ax.set_title('Comparación: w del vacío DEE con casos cosmológicos', fontsize=12)
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
save_plot('38_ecuacion_estado')
plt.show()

## Análisis de relaciones internas

Ahora exploramos si hay relaciones cuantitativas entre las cantidades que sugieran principios de equilibrio.

In [ ]:
print('='*75)
print('RELACIONES TERMODINÁMICAS EN EL ESTADO FUNDAMENTAL DEE')
print('='*75)
print()
print(f'Cantidades primarias:')
print(f'  ρ_E    (densidad de energía):     {rho_E:.4f}')
print(f'  P_vac  (presión del vacío):       {P_vacuum:.4f}')
print(f'  c_s²   (velocidad de sonido²):    {c_s_natural**2:.4f}')
if K_bulk is not None:
    print(f'  K      (módulo de bulk):          {K_bulk:.4f}')
print(f'  σ²     (tensión cuántica/nodo):   {sigma_sq_mean:.4f}')
print()

print(f'Razones e identidades:')
print(f'  P/ρ = {P_vacuum/rho_E:.4f}')
print(f'  P/(ρc²) = w = {w_eos:.4f}')
if K_bulk is not None:
    print(f'  K/(ρc²) = {K_bulk/(rho_E*c_s_natural**2):.4f}  (debería ser ≈1)')
print(f'  σ² · ρ_E = {sigma_sq_mean*rho_E:.4f}')
print(f'  σ² / a²  = {sigma_sq_mean/spacing_typical**2:.4f}  (a = espaciado típico)')
print()

# Tu intuición: ¿hay relación inversa entre tensión y densidad?
print(f'Test de relación inversa propuesta (σ² ∝ 1/ρ):')
product = sigma_sq_mean * rho_E
print(f'  σ² × ρ = {product:.4f}')
print(f'  Si fuera constante en distintos volúmenes, sería verificación')

In [ ]:
# Verificación de la relación σ²·ρ entre los volúmenes calculados
# (ya tenemos E_0 en distintos V, podemos calcular σ² en cada uno)

print('Test de relación σ²·ρ a través de distintos volúmenes:')
print(f'{"ε":>6} {"V":>8} {"E₀/V":>10} {"⟨σ²⟩":>10} {"σ²·ρ":>10}')
print('-'*50)

sigmas_test = []
rhos_test = []

for eps_idx, eps in enumerate(epsilons):
    L_box = 1.0 + eps
    points_eps = points * L_box
    D_eps = periodic_distance_matrix(points_eps, L=L_box)
    L_op_eps = build_laplacian(D_eps, K_NEIGHBORS, alpha=2.0)
    eigs_eps, vecs_eps = eigsh(L_op_eps, k=400, which='SM', tol=1e-8)
    idx_s = np.argsort(eigs_eps)
    eigs_eps = eigs_eps[idx_s]; vecs_eps = vecs_eps[:, idx_s]
    eigs_nz = eigs_eps[eigs_eps > 1e-3]
    vecs_nz = vecs_eps[:, eigs_eps > 1e-3]
    omegas_eps = np.sqrt(eigs_nz)
    
    weights_eps = 1.0 / (2 * omegas_eps)
    sigma_sq_eps = np.mean(np.sum(vecs_nz**2 * weights_eps[None, :], axis=1))
    rho_eps = np.sum(0.5 * omegas_eps) / L_box**3
    
    sigmas_test.append(sigma_sq_eps)
    rhos_test.append(rho_eps)
    
    print(f'{eps:+.3f} {L_box**3:>8.4f} {rho_eps:>10.4f} {sigma_sq_eps:>10.4f} '
          f'{sigma_sq_eps*rho_eps:>10.4f}')

sigmas_test = np.array(sigmas_test)
rhos_test = np.array(rhos_test)
products = sigmas_test * rhos_test

print()
if products.std() / products.mean() < 0.05:
    print(f'→ σ²·ρ es aproximadamente constante: {products.mean():.4f} ± {products.std():.4f}')
    print(f'  ✓ Tu intuición sobre relación inversa SE CONFIRMA')
else:
    print(f'→ σ²·ρ varía: {products.mean():.4f} ± {products.std():.4f}')
    print(f'  Coeficiente de variación: {products.std()/products.mean()*100:.1f}%')
    print(f'  La relación inversa pura no se cumple, pero puede haber variante')

In [ ]:
# Plot final: σ² vs ρ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(rhos_test, sigmas_test, 'o-', markersize=12, lw=2, color='darkblue',
        label='Medido')
# Hipótesis inversa: σ² = C/ρ con C = producto promedio
rho_smooth = np.linspace(rhos_test.min()*0.9, rhos_test.max()*1.1, 50)
ax.plot(rho_smooth, products.mean()/rho_smooth, '--', color='red', lw=2,
        label=f'σ² = {products.mean():.3f}/ρ (hipótesis inversa)')
ax.set_xlabel('ρ (densidad de energía)', fontsize=12)
ax.set_ylabel('σ² (tensión cuántica)', fontsize=12)
ax.set_title('Relación entre tensión cuántica y densidad', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(np.array(V_vs_eps), products, 'o-', markersize=12, lw=2, color='darkgreen')
ax.axhline(products.mean(), color='red', linestyle='--', 
           label=f'Promedio: {products.mean():.4f}')
ax.set_xlabel('V (volumen)', fontsize=12)
ax.set_ylabel('σ² · ρ (producto)', fontsize=12)
ax.set_title('Producto σ²·ρ vs V — ¿es invariante?', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('39_relacion_tension_densidad')
plt.show()

In [ ]:
print('='*75)
print('SÍNTESIS — Termodinámica del estado fundamental DEE')
print('='*75)
print()
print('Caracterización completa del vacío cristalino:')
print(f'  1. Tensión cuántica residual:    σ² = {sigma_sq_mean:.4f} por nodo')
print(f'     (amplitud típica σ = {sigma_typical:.4f}, vs espaciado a = {spacing_typical:.4f})')
print(f'     σ/a = {sigma_typical/spacing_typical:.3f}: fluctuaciones cuánticas')
print(f'     {"PEQUEÑAS comparadas con espaciado (régimen armónico válido)" if sigma_typical/spacing_typical < 1 else "GRANDES (cristal cuántico fuertemente fluctuante)"}')
print()
print(f'  2. Densidad de energía:          ρ_E = {rho_E:.4f} (unidades naturales)')
print(f'     Por nodo:                     {E_0_per_node_natural:.4f}')
print()
print(f'  3. Presión del vacío:            P_vac = {P_vacuum:+.4f}')
print(f'     Ecuación de estado:           w = {w_eos:+.4f}')
if w_eos < -0.5:
    print(f'     → Comportamiento tipo "energía oscura" (w < -1/3)')
elif abs(w_eos) < 0.1:
    print(f'     → Comportamiento tipo "polvo" (w ≈ 0)')
elif w_eos > 0.2:
    print(f'     → Comportamiento tipo "radiación" (w > 0)')
print()
if K_bulk is not None:
    print(f'  4. Módulo de bulk:               K = {K_bulk:.4f}')
    print(f'     Verificación K = ρc²:         {K_bulk/(rho_E*c_s_natural**2):.3f} (debería ser ~1)')
print()
print('Relación tensión ↔ densidad:')
print(f'  σ²·ρ = {products.mean():.4f} ± {products.std():.4f}')
if products.std()/products.mean() < 0.05:
    print(f'  → Producto APROXIMADAMENTE CONSTANTE (variación {products.std()/products.mean()*100:.1f}%)')
    print(f'    Tu intuición de relación inversa se confirma cuantitativamente.')
else:
    print(f'  → Producto VARÍA (coef. variación {products.std()/products.mean()*100:.1f}%)')
    print(f'    La relación es más compleja que σ² ∝ 1/ρ exacto.')

In [ ]:
import shutil
shutil.make_archive('plots_termo_vacio', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_termo_vacio.zip')

try:
    from google.colab import files
    files.download('plots_termo_vacio.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass